# Validate the notebook controller

Verify controller isolation and exact notebook-tooling versions. Optionally probe the configured scientific runtime to confirm it still matches its separate lock. The controller is not yet a training kernel.


In [ ]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)


## Controller identity and package pins


In [ ]:
import importlib.metadata as metadata
import re
settings = execution_settings(ROOT)
assert f'{sys.version_info.major}.{sys.version_info.minor}' == settings['controller_python']
assert sys.prefix != sys.base_prefix, 'Use the isolated controller virtual environment'
tooling = json.loads((ROOT / 'environment/notebook_tooling/installed_packages.json').read_text())
actual = {re.sub(r'[-_.]+', '-', d.metadata['Name']).lower(): d.version for d in metadata.distributions()}
assert actual == tooling, {'missing': sorted(tooling.keys() - actual.keys()), 'extra': sorted(actual.keys() - tooling.keys()), 'changed': {k: [tooling[k],actual[k]] for k in tooling.keys() & actual.keys() if tooling[k] != actual[k]}}
for name in ['torch', 'stable-baselines3', 'mlagents-envs']:
    assert name not in actual, 'Scientific packages unexpectedly installed in controller'
check = subprocess.run([sys.executable, '-m', 'pip', 'check'], capture_output=True, text=True, timeout=90)
assert check.returncode == 0, check.stdout
print('Controller:', sys.executable)
print('Pinned distributions:', len(actual))


## Optional runtime parity check


In [ ]:
assets = load_assets(ROOT)
state = next(row for row in asset_status(assets) if row['asset'] == 'runtime_python')
if state['status'] == 'AVAILABLE':
    runtime = require_asset(assets, 'runtime_python')
    assert runtime.resolve() != Path(sys.executable).resolve(), 'Controller and runtime must be distinct'
    runtime_result = compare_runtime(ROOT, runtime_probe(runtime))
    assert runtime_result['status'] == 'PASS', runtime_result
else:
    runtime_result = {'status': 'NOT_RUN', 'reason': state['status']}
result = {'controller_status': 'PASS', 'controller_distributions': len(actual), 'runtime': runtime_result,
          'scope': 'Controller tooling and optional runtime package/CPU probe; no notebook-based model/training parity claim.'}
print(json.dumps(result, indent=2))
print('Local evidence:', save_record(ROOT, 'notebook_environment', result))
